<a href="https://colab.research.google.com/github/saim873/flyrank-ml-internship-Saim/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Setup Complete")

Setup Complete


In [6]:
df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Number of clients:", df["client_id"].nunique())

df.head()

Dataset shape: (30000, 44)
Number of clients: 32


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1
The paper reports improved prediction performance.

**My methodology question:**
Was the evaluation performed using a grouped or time-aware validation split to avoid data leakage?

---

## Finding 2
The paper compares the proposed model with a baseline.

**My methodology question:**
Were the same dataset, validation split, and evaluation metric used for both models to ensure a fair comparison?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [7]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

# -------------------------------------------------
# 1. Prepare an observed binary label
# -------------------------------------------------

model_df = df[
    df["trend_direction"].isin(["down", "stable", "up"])
].copy()

model_df["is_declining_label"] = (
    model_df["trend_direction"] == "down"
).astype(int)

# Leakage and identifier columns are excluded
drop_columns = [
    "content_id",
    "client_id",
    "trend_pct",
    "trend_direction",
    "is_declining_label"
]

X = model_df.drop(columns=drop_columns)
y = model_df["is_declining_label"]
groups = model_df["client_id"]

# -------------------------------------------------
# 2. Preprocessing
# -------------------------------------------------

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

def build_model():
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=200,
                    random_state=RANDOM_STATE,
                    class_weight="balanced",
                    n_jobs=-1
                )
            )
        ]
    )

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    return {
        "Validation design": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "Test rows": len(y_test),
        "Test base rate": y_test.mean()
    }, predictions, model

# -------------------------------------------------
# 3. BEFORE: ordinary random row split
# -------------------------------------------------

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y
    )
)

random_result, random_predictions, random_model = evaluate_model(
    "Before: random row split",
    build_model(),
    X_train_random,
    X_test_random,
    y_train_random,
    y_test_random
)

# -------------------------------------------------
# 4. AFTER: grouped split by client
# -------------------------------------------------

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    group_splitter.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx].copy()
X_test_grouped = X.iloc[test_idx].copy()

y_train_grouped = y.iloc[train_idx].copy()
y_test_grouped = y.iloc[test_idx].copy()

grouped_result, grouped_predictions, grouped_model = evaluate_model(
    "After: grouped by client",
    build_model(),
    X_train_grouped,
    X_test_grouped,
    y_train_grouped,
    y_test_grouped
)

# -------------------------------------------------
# 5. Before/after comparison
# -------------------------------------------------

validation_comparison = pd.DataFrame(
    [random_result, grouped_result]
)

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Test base rate"
]

validation_comparison[metric_columns] = (
    validation_comparison[metric_columns].round(3)
)

print(
    "Random-split test clients:",
    groups.loc[X_test_random.index].nunique()
)

print(
    "Grouped-split train clients:",
    groups.iloc[train_idx].nunique()
)

print(
    "Grouped-split test clients:",
    groups.iloc[test_idx].nunique()
)

print(
    "Client overlap in grouped split:",
    len(
        set(groups.iloc[train_idx])
        .intersection(set(groups.iloc[test_idx]))
    )
)

validation_comparison

Random-split test clients: 30
Grouped-split train clients: 24
Grouped-split test clients: 7
Client overlap in grouped split: 0


,Validation design,Accuracy,Precision,Recall,F1,Test rows,Test base rate
0,Before: random row split,0.841,0.838,0.916,0.875,5323,0.611
1,After: grouped by client,0.796,0.812,0.871,0.840,5098,0.617


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
leakage_columns = [
    "trend_pct",
    "trend_direction"
]

identifier_columns = [
    "content_id",
    "client_id"
]

print("Leakage Features")
for col in leakage_columns:
    print("-", col)

print("\nIdentifier Features")
for col in identifier_columns:
    print("-", col)

print("\nAudit Result")
print("✓ trend_pct removed")
print("✓ trend_direction removed")
print("✓ content_id removed")
print("✓ client_id removed")
print("✓ No future information used")
print("✓ No product flags used")

Leakage Features
- trend_pct
- trend_direction

Identifier Features
- content_id
- client_id

Audit Result
✓ trend_pct removed
✓ trend_direction removed
✓ content_id removed
✓ client_id removed
✓ No future information used
✓ No product flags used


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim rewrite

### Original claim
The Random Forest model is highly accurate and can identify declining content reliably.

### Revised claim
On the grouped client-level test split, the Random Forest model measured an accuracy of 0.796 and an F1 score of 0.840.

These results indicate that the model provides useful directional decision-support for identifying content that may be declining. However, the findings do not prove causation and may not generalize equally to every client, content type, or future time period.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

- ✅ Every section is filled with markdown and supporting code.
- ✅ The notebook runs from top to bottom without errors.
- ✅ No client names, URLs, or private queries are included.
- ✅ Leakage-prone columns and identifiers were excluded from the feature set.
- ✅ The grouped split contains zero client overlap.
- ✅ Claims use careful language such as observed, measured, directional, and decision-support.
- ✅ The notebook will be committed under `work/notebooks/w06_validation_audit.ipynb`.